In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

## Load Sample Data

Load price data from the parquet file. We'll use a few tickers as examples.

In [2]:
# Load the parquet data
data_path = Path("../../equities/sp500_daily")

if data_path.exists():
    df = pd.read_parquet(data_path)
    
    # Normalize column names
    df.columns = [str(c).strip().lower() for c in df.columns]
    
    # Ensure date is datetime and set as index
    if 'date' in df.columns:
        df['date'] = pd.to_datetime(df['date'])
        df = df.set_index('date')
    
    print(f"Loaded data shape: {df.shape}")
    print(f"Date range: {df.index.min()} to {df.index.max()}")
    print(f"Columns: {df.columns.tolist()}")
    print(f"\nSample tickers: {df['ticker'].unique()[:10].tolist()}")
else:
    print(f"Data path not found: {data_path}")
    print("Creating synthetic data for demonstration...")
    
    # Create synthetic data
    dates = pd.date_range('2020-01-01', '2024-12-31', freq='B')
    tickers = ['AAPL', 'MSFT', 'GOOGL']
    
    data = []
    for ticker in tickers:
        np.random.seed(hash(ticker) % 2**32)
        prices = 100 * np.exp(np.cumsum(np.random.randn(len(dates)) * 0.02))
        for date, price in zip(dates, prices):
            data.append({
                'date': date,
                'ticker': ticker,
                'close': price,
                'volume': np.random.randint(1_000_000, 10_000_000)
            })
    
    df = pd.DataFrame(data)
    df = df.set_index('date')
    print("Created synthetic data for demonstration")

Data path not found: ..\..\equities\sp500_daily
Creating synthetic data for demonstration...
Created synthetic data for demonstration


## Select Tickers and Prepare Data

Select a few tickers to analyze and create wide-format dataframes.

In [3]:
# Select tickers to display
sample_tickers = df['ticker'].unique()[:3].tolist()
print(f"Analyzing tickers: {sample_tickers}")

# Filter data for selected tickers
df_filtered = df[df['ticker'].isin(sample_tickers)].copy()

# Create wide format for prices
prices = df_filtered.pivot(columns='ticker', values='close')
prices = prices.ffill()  # Forward fill missing values

print(f"\nPrice data shape: {prices.shape}")
print(f"Date range: {prices.index.min()} to {prices.index.max()}")
prices.tail()

Analyzing tickers: ['AAPL', 'MSFT', 'GOOGL']

Price data shape: (1305, 3)
Date range: 2020-01-01 00:00:00 to 2024-12-31 00:00:00


ticker,AAPL,GOOGL,MSFT
date,,,
2024-12-25,120.745417,53.493123,258.068455
2024-12-26,119.912606,57.005976,257.460825
2024-12-27,115.733620,56.609370,263.591440
2024-12-30,119.183735,55.338430,267.697430
2024-12-31,118.832640,55.244336,268.774881


## Generate Simulated Signals and Positions

For demonstration, we'll create:
- **Signals**: Momentum-based signals (6-month returns ranked)
- **Positions**: Long/short positions based on signals

In [4]:
# Calculate 6-month momentum signal
lookback = 126  # ~6 months of trading days
returns_126 = prices.pct_change(lookback)

# Rank signals cross-sectionally (1 = lowest, higher = better momentum)
signals = returns_126.rank(axis=1, pct=True)

# Create positions based on signals
# Long top 1/3, short bottom 1/3, neutral middle
positions = pd.DataFrame(0.0, index=signals.index, columns=signals.columns)
positions[signals > 0.67] = 1.0   # Long
positions[signals < 0.33] = -1.0  # Short

print("Signals (momentum rank):")
print(signals.tail())
print("\nPositions (1=long, -1=short, 0=neutral):")
print(positions.tail())

Signals (momentum rank):
ticker          AAPL  GOOGL      MSFT
date                                 
2024-12-25  0.666667    1.0  0.333333
2024-12-26  0.666667    1.0  0.333333
2024-12-27  0.666667    1.0  0.333333
2024-12-30  0.666667    1.0  0.333333
2024-12-31  0.666667    1.0  0.333333

Positions (1=long, -1=short, 0=neutral):
ticker      AAPL  GOOGL  MSFT
date                         
2024-12-25   0.0    1.0   0.0
2024-12-26   0.0    1.0   0.0
2024-12-27   0.0    1.0   0.0
2024-12-30   0.0    1.0   0.0
2024-12-31   0.0    1.0   0.0


## Calculate Asset Returns

Normalize prices to percentage returns for better visualization.

In [5]:
# Calculate cumulative returns (indexed to 100)
returns = prices.pct_change()
cum_returns = (1 + returns).cumprod() * 100

# Use recent data for better visualization (last 2 years)
start_date = prices.index.max() - pd.DateOffset(years=2)
prices_plot = prices[prices.index >= start_date]
signals_plot = signals[signals.index >= start_date]
positions_plot = positions[positions.index >= start_date]
cum_returns_plot = cum_returns[cum_returns.index >= start_date]

print(f"Plotting data from {prices_plot.index.min()} to {prices_plot.index.max()}")
print(f"Number of trading days: {len(prices_plot)}")

Plotting data from 2023-01-02 00:00:00 to 2024-12-31 00:00:00
Number of trading days: 522


## Create Synchronized Charts

Three vertically stacked charts with shared x-axis and synchronized hover.

In [ ]:
# Create subplots with shared x-axis
fig = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
 

## Alternative: Single Chart with Dropdowns

Create an interactive chart where you can select which ticker to view across all three panels.

In [7]:
# Create figure with dropdowns for single ticker selection
selected_ticker = sample_tickers[0]

fig2 = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=(
        f'{selected_ticker} - Cumulative Returns',
        f'{selected_ticker} - Trading Signal',
        f'{selected_ticker} - Position'
    ),
    row_heights=[0.4, 0.3, 0.3]
)

# Add traces for each ticker (initially all visible for first ticker)
for ticker_idx, ticker in enumerate(sample_tickers):
    visible = (ticker == selected_ticker)
    
    # Price trace
    fig2.add_trace(
        go.Scatter(
            x=cum_returns_plot.index,
            y=cum_returns_plot[ticker],
            name=f"{ticker} Returns",
            mode='lines',
            line=dict(color='#1f77b4', width=3),
            visible=visible,
            hovertemplate='Date: %{x|%Y-%m-%d}<br>Return: %{y:.2f}<extra></extra>'
        ),
        row=1, col=1
    )
    
    # Signal trace
    fig2.add_trace(
        go.Scatter(
            x=signals_plot.index,
            y=signals_plot[ticker],
            name=f"{ticker} Signal",
            mode='lines',
            line=dict(color='#ff7f0e', width=3),
            visible=visible,
            fill='tozeroy',
            fillcolor='rgba(255, 127, 14, 0.1)',
            hovertemplate='Date: %{x|%Y-%m-%d}<br>Signal: %{y:.3f}<extra></extra>'
        ),
        row=2, col=1
    )
    
    # Position trace
    fig2.add_trace(
        go.Scatter(
            x=positions_plot.index,
            y=positions_plot[ticker],
            name=f"{ticker} Position",
            mode='lines',
            line=dict(color='#2ca02c', width=3, shape='hv'),
            visible=visible,
            fill='tozeroy',
            fillcolor='rgba(44, 160, 44, 0.2)',
            hovertemplate='Date: %{x|%Y-%m-%d}<br>Position: %{y:.0f}<extra></extra>'
        ),
        row=3, col=1
    )

# Add reference lines to position chart
for y_val in [1, 0, -1]:
    fig2.add_hline(y=y_val, line_dash="dash", line_color="gray", opacity=0.3, row=3, col=1)

# Create dropdown buttons
buttons = []
for ticker_idx, ticker in enumerate(sample_tickers):
    # Each ticker has 3 traces (returns, signal, position)
    visible_array = [False] * (len(sample_tickers) * 3)
    visible_array[ticker_idx * 3] = True      # Returns
    visible_array[ticker_idx * 3 + 1] = True  # Signal
    visible_array[ticker_idx * 3 + 2] = True  # Position
    
    buttons.append(
        dict(
            label=ticker,
            method="update",
            args=[
                {"visible": visible_array},
                {"title": f"Trading Analysis - {ticker}"}
            ]
        )
    )

# Update layout
fig2.update_layout(
    height=900,
    title_text=f"Trading Analysis - {selected_ticker}",
    title_x=0.5,
    title_font_size=20,
    hovermode='x unified',
    template='plotly_white',
    showlegend=False,
    updatemenus=[
        dict(
            active=0,
            buttons=buttons,
            direction="down",
            pad={"r": 10, "t": 10},
            showactive=True,
            x=0.02,
            xanchor="left",
            y=1.15,
            yanchor="top"
        )
    ],
    annotations=[
        dict(
            text="Select Ticker:",
            showarrow=False,
            x=0.0,
            y=1.18,
            xref="paper",
            yref="paper",
            align="left",
            font=dict(size=14)
        )
    ]
)

# Update axes
fig2.update_yaxes(title_text="Cumulative Return", row=1, col=1)
fig2.update_yaxes(title_text="Signal Rank (0-1)", row=2, col=1)
fig2.update_yaxes(title_text="Position", row=3, col=1, range=[-1.5, 1.5])
fig2.update_xaxes(title_text="Date", row=3, col=1)

fig2.show()

## Summary

This notebook demonstrates:

1. **Synchronized hover**: Using `hovermode='x unified'` allows the crosshair to show data from all three charts at the same date
2. **Shared x-axis**: All charts zoom and pan together
3. **Multiple visualization approaches**: 
   - Multiple tickers on same chart (good for comparison)
   - Single ticker with dropdown selector (good for detailed analysis)

### Key Plotly Features Used:
- `make_subplots()` with `shared_xaxes=True`
- `hovermode='x unified'` for synchronized crosshair
- `updatemenus` for interactive dropdown selection
- `legendgroup` to link traces across subplots

### Possible Enhancements:
- Add range slider for easy time range selection
- Include additional metrics (Sharpe ratio, drawdown, etc.)
- Add portfolio-level aggregated returns
- Export to HTML for sharing